# AstroCLIMB — Qwen3-VL-8B language-attention QLoRA

This notebook trains **Qwen3-VL-8B-Instruct** on all 10,000 labeled rows using 4-bit NF4 QLoRA on the language attention projections (`q_proj`, `k_proj`, `v_proj`, and `o_proj`). It preserves the original label-token loss used by the successful full-10K 4B baseline.

It is designed for a clean Kaggle **T4 ×2** session. Training and inference use two fresh Accelerate workers, one quantized model replica per GPU. Target-module checks stop the run if a visual module is accidentally selected.


In [ ]:
# Preserve Kaggle's torch, torchvision, Pillow, and scikit-learn versions.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes==0.47.0"

In [ ]:
import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)
print('torch:', torch.__version__)
print('CUDA has not been initialized:', not torch.cuda.is_initialized())

In [ ]:
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
NUM_EPOCHS = 1
GRADIENT_ACCUMULATION = 8  # global batch: 1 x 2 GPUs x 8 = 16
OVERSAMPLE_SAME_FIGURE = False  # True includes all 10K rows plus 2K repeated class-0 rows.
REBUILD_CACHE = False
RUN_TEST_INFERENCE = True
TEST_LIMIT = None  # None produces all 10,000 predictions; use 100 for a timing check.
USE_SWAP_TTA = False
APPLY_MODALITY_MASK = False

WORK_ROOT = Path('/kaggle/working/astroclimb_qwen3vl8b_language') if Path('/kaggle/working').exists() else Path('./astroclimb_qwen3vl8b_language')
IMAGE_ROOT = WORK_ROOT / 'images_448'
TRAIN_MANIFEST = WORK_ROOT / 'train_10000.jsonl'
TEST_MANIFEST = WORK_ROOT / 'test_10000.jsonl'
ADAPTER_DIR = WORK_ROOT / 'adapter_final'
TRAINER_DIR = WORK_ROOT / 'trainer_output'
PREDICTION_DIR = WORK_ROOT / 'test_shards'
for path in (WORK_ROOT, IMAGE_ROOT, PREDICTION_DIR):
    path.mkdir(parents=True, exist_ok=True)

def locate_csv(filename):
    for directory in (Path('/kaggle/input/competitions/astroclimb'), Path('/kaggle/input/astroclimb')):
        candidate = directory / filename
        if candidate.exists():
            return candidate
    roots = [Path('/kaggle/input'), Path('data')]
    candidates = [p for root in roots if root.exists() for p in root.rglob(filename)]
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'{filename} not found.')
    return candidates[0]

def locate_model():
    if Path('/kaggle/input').exists():
        configs = list(Path('/kaggle/input').rglob('config.json'))
        models = [
            p.parent for p in configs
            if 'qwen3' in str(p).lower() and 'vl' in str(p).lower() and '8b' in str(p).lower()
        ]
        if models:
            return str(sorted(models, key=lambda p: len(str(p)))[0])
    return MODEL_ID

TRAIN_CSV = locate_csv('train.csv')
TEST_CSV = locate_csv('test.csv')
MODEL_PATH = locate_model()
print('Train:', TRAIN_CSV)
print('Test:', TEST_CSV)
print('Model:', MODEL_PATH)
print('Output:', WORK_ROOT)

## Stream and cache all train/test objects

The original CSVs are never loaded into RAM. Each unique image is decoded, resized to the 448² area budget, and cached once. JSONL manifests contain only captions or local image paths. This stage can require several gigabytes of working storage.

In [ ]:
def looks_like_image(value):
    if not isinstance(value, str):
        return False
    return value.lstrip().startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image'))

def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    image = Image.open(io.BytesIO(base64.b64decode(value, validate=False)))
    image.load()
    return image.convert('RGB')

def resize_to_area(image, max_pixels=MAX_PIXELS):
    width, height = image.size
    if width * height <= max_pixels:
        return image
    scale = math.sqrt(max_pixels / (width * height))
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return image.resize(size, Image.Resampling.LANCZOS)

def cache_object(value):
    if not looks_like_image(value):
        return {'kind': 'caption', 'value': value}
    digest = hashlib.sha256(value.encode('utf-8')).hexdigest()
    destination = IMAGE_ROOT / f'{digest}.png'
    if not destination.exists():
        image = resize_to_area(decode_image(value))
        image.save(destination, format='PNG', compress_level=3)
    return {'kind': 'image', 'value': str(destination)}

def get_label(row):
    values = [int(float(row[c])) for c in TARGET_COLUMNS]
    if sum(values) != 1:
        raise ValueError(f'Invalid target for id={row.get("id")}: {values}')
    return values.index(1)

def build_manifest(source, destination, labeled):
    started = time.perf_counter()
    count = 0
    class_counts = np.zeros(4, dtype=int)
    modality_counts = {}
    with (
        source.open('r', encoding='utf-8', newline='') as src,
        destination.open('w', encoding='utf-8') as dst,
    ):
        reader = csv.DictReader(src)
        required = {'id', 'obj_1', 'obj_2'} | (set(TARGET_COLUMNS) if labeled else set())
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'{source.name} missing columns: {sorted(missing)}')
        for row in reader:
            obj_1, obj_2 = cache_object(row['obj_1']), cache_object(row['obj_2'])
            mode = obj_1['kind'][0].upper() + obj_2['kind'][0].upper()
            record = {'id': row['id'], 'obj_1': obj_1, 'obj_2': obj_2, 'modality': mode}
            if labeled:
                record['label'] = get_label(row)
                class_counts[record['label']] += 1
            dst.write(json.dumps(record, ensure_ascii=False) + '\n')
            modality_counts[mode] = modality_counts.get(mode, 0) + 1
            count += 1
            if count % 500 == 0:
                print(f'{source.name}: {count} rows | {(time.perf_counter() - started) / 60:.1f} min')
    print(source.name, 'rows=', count, 'classes=', class_counts.tolist(), 'modalities=', modality_counts)
    print(f'{destination.name} completed in {(time.perf_counter() - started) / 60:.2f} min')
    return count

if REBUILD_CACHE or not TRAIN_MANIFEST.exists():
    train_count = build_manifest(TRAIN_CSV, TRAIN_MANIFEST, labeled=True)
else:
    train_count = sum(1 for _ in TRAIN_MANIFEST.open(encoding='utf-8'))
    print('Reusing:', TRAIN_MANIFEST)
if REBUILD_CACHE or not TEST_MANIFEST.exists():
    test_count = build_manifest(TEST_CSV, TEST_MANIFEST, labeled=False)
else:
    test_count = sum(1 for _ in TEST_MANIFEST.open(encoding='utf-8'))
    print('Reusing:', TEST_MANIFEST)
assert train_count == 10000 and test_count == 10000
print('Unique cached images:', len(list(IMAGE_ROOT.glob('*.png'))))
print(f'Image cache size: {sum(p.stat().st_size for p in IMAGE_ROOT.glob("*.png")) / 2**30:.2f} GiB')

## Prompt and training dataset

In [ ]:
SYSTEM_PROMPT = '''You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.'''.strip()

def shorten_caption(text):
    if len(text) <= MAX_TEXT_CHARS:
        return text
    half = MAX_TEXT_CHARS // 2
    return text[:half] + '\n[...middle truncated...]\n' + text[-half:]

def object_content(number, obj):
    if obj['kind'] == 'image':
        with Image.open(obj['value']) as source:
            image = source.convert('RGB')
        return [
            {'type': 'text', 'text': f'Object {number} is a scientific figure:'},
            {'type': 'image', 'image': image},
        ]
    return [{'type': 'text', 'text': f'Object {number} is a figure caption:\n{shorten_caption(obj["value"])}'}]

def build_messages(row, include_answer, swap=False):
    obj_1, obj_2 = row['obj_1'], row['obj_2']
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({'type': 'text', 'text': 'Classify their relationship. Reply with one digit only.'})
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': content},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': str(int(row['label']))}]})
    return messages

class ManifestDataset(torch.utils.data.Dataset):
    def __init__(self, path, training=False, oversample_same_figure=False):
        with Path(path).open('r', encoding='utf-8') as handle:
            self.rows = [json.loads(line) for line in handle]
        if training and oversample_same_figure:
            class_zero = [row for row in self.rows if row['label'] == 0]
            self.rows.extend(class_zero)
            self.rows.extend(class_zero)
        self.training = training
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = dict(self.rows[index])
        row['_swap'] = self.training and random.random() < 0.5
        return row

class LabelOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        self.label_token_ids = []
        for digit in '0123':
            ids = processor.tokenizer.encode(digit, add_special_tokens=False)
            if len(ids) != 1:
                raise ValueError(f'Label {digit} is not one token: {ids}')
            self.label_token_ids.append(ids[0])
    def __call__(self, features):
        if len(features) != 1:
            raise ValueError(f'Per-device batch must be 1; received {len(features)}')
        row = features[0]
        batch = self.processor.apply_chat_template(
            build_messages(row, include_answer=True, swap=row['_swap']),
            tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt',
        )
        target_id = self.label_token_ids[int(row['label'])]
        positions = torch.where(batch['input_ids'][0] == target_id)[0]
        if not len(positions):
            raise RuntimeError('Assistant label token was not found.')
        labels = torch.full_like(batch['input_ids'], -100)
        labels[0, int(positions[-1])] = target_id
        batch['labels'] = labels
        return batch

## Train Qwen3-VL-8B language-attention LoRA on both T4s

Only the language attention projections are adapted. The worker prints and validates every trainable parameter before training. With global batch 16, one full epoch is approximately 625 optimizer steps.


In [ ]:
def train_worker(manifest_path, model_path, adapter_path, oversample):
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    rank = int(os.environ.get('LOCAL_RANK', '0'))
    torch.cuda.set_device(rank)
    random.seed(SEED + rank); np.random.seed(SEED + rank); torch.manual_seed(SEED + rank)
    load_started = time.perf_counter()
    processor = AutoProcessor.from_pretrained(model_path, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    processor.tokenizer.padding_side = 'right'
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_path, quantization_config=quantization, dtype=torch.float16,
        attn_implementation='sdpa', device_map={'': rank},
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ))
    dataset = ManifestDataset(manifest_path, training=True, oversample_same_figure=oversample)
    if rank == 0:
        model.print_trainable_parameters()
        print('Effective training rows:', len(dataset))
        print(f'Model load: {(time.perf_counter() - load_started) / 60:.2f} min')

    args = TrainingArguments(
        output_dir=str(TRAINER_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS, learning_rate=1e-4, warmup_ratio=0.05,
        lr_scheduler_type='cosine', weight_decay=0.01, max_grad_norm=1.0,
        fp16=True, bf16=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit', logging_steps=10,
        eval_strategy='no', save_strategy='steps', save_steps=100, save_total_limit=2,
        report_to='none', remove_unused_columns=False, dataloader_num_workers=0,
        ddp_find_unused_parameters=False, seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=LabelOnlyCollator(processor))
    torch.cuda.synchronize(); started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize(); elapsed = time.perf_counter() - started
    if trainer.is_world_process_zero():
        destination = Path(adapter_path); destination.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(destination); processor.save_pretrained(destination)
        metrics = dict(result.metrics)
        metrics.update({
            'wall_minutes': elapsed / 60,
            'optimizer_steps': int(trainer.state.global_step),
            'seconds_per_step': elapsed / max(1, trainer.state.global_step),
            'peak_gpu_gib_rank0': torch.cuda.max_memory_allocated() / 2**30,
            'effective_training_rows': len(dataset),
        })
        with (destination / 'training_metrics.json').open('w') as handle:
            json.dump(metrics, handle, indent=2)
        print(json.dumps(metrics, indent=2))

# Launch a self-contained script in fresh interpreters (safe even if the kernel touched CUDA).
TRAIN_SCRIPT_PATH = WORK_ROOT / 'train_ddp.py'
TRAIN_SCRIPT_PATH.write_bytes(base64.b64decode('aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UsIEltYWdlRmlsZQoKSW1hZ2VGaWxlLkxPQURfVFJVTkNBVEVEX0lNQUdFUyA9IFRydWUKClNFRUQgPSA0MgpNSU5fUElYRUxTID0gMjU2ICogMjU2Ck1BWF9QSVhFTFMgPSA0NDggKiA0NDgKTUFYX1RFWFRfQ0hBUlMgPSAzMDAwCgpTWVNURU1fUFJPTVBUID0gIiIiWW91IGNsYXNzaWZ5IHRoZSByZWxhdGlvbnNoaXAgYmV0d2VlbiB0d28gb2JqZWN0cyBmcm9tIGFzdHJvbm9teSBwYXBlcnMuCjA6IFRoZSBvYmplY3RzIGFyZSB0aGUgZmlndXJlIGFuZCBjYXB0aW9uIG9mIHRoZSBzYW1lIHNjaWVudGlmaWMgZmlndXJlLgoxOiBUaGUgb2JqZWN0cyBhcmUgZnJvbSBkaWZmZXJlbnQgZmlndXJlcyBpbiB0aGUgc2FtZSBwYXBlci4KMjogVGhlIG9iamVjdHMgYXJlIGZyb20gZGlmZmVyZW50IHBhcGVycyBhbmQgb25lIHBhcGVyIGNpdGVzIHRoZSBvdGhlci4KMzogVGhlIG9iamVjdHMgYXJlIGZyb20gdW5yZWxhdGVkIHBhcGVycy4KVGhlIHJlbGF0aW9uc2hpcCBpcyBzeW1tZXRyaWMuIE91dHB1dCBvbmx5IG9uZSBkaWdpdDogMCwgMSwgMiwgb3IgMy4iIiIuc3RyaXAoKQoKCmRlZiBzaG9ydGVuX2NhcHRpb24odGV4dCwgbWF4X2NoYXJzPU1BWF9URVhUX0NIQVJTKToKICAgIGlmIGxlbih0ZXh0KSA8PSBtYXhfY2hhcnM6CiAgICAgICAgcmV0dXJuIHRleHQKICAgIGhhbGYgPSBtYXhfY2hhcnMgLy8gMgogICAgcmV0dXJuIHRleHRbOmhhbGZdICsgIlxuWy4uLm1pZGRsZSB0cnVuY2F0ZWQuLi5dXG4iICsgdGV4dFstaGFsZjpdCgoKZGVmIG9iamVjdF9jb250ZW50KG51bWJlciwgb2JqKToKICAgIGlmIG9ialsia2luZCJdID09ICJpbWFnZSI6CiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKG9ialsidmFsdWUiXSkgYXMgc291cmNlOgogICAgICAgICAgICBpbWFnZSA9IHNvdXJjZS5jb252ZXJ0KCJSR0IiKQogICAgICAgIHJldHVybiBbCiAgICAgICAgICAgIHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBmIk9iamVjdCB7bnVtYmVyfSBpcyBhIHNjaWVudGlmaWMgZmlndXJlOiJ9LAogICAgICAgICAgICB7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWFnZX0sCiAgICAgICAgXQogICAgcmV0dXJuIFt7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogZiJPYmplY3Qge251bWJlcn0gaXMgYSBmaWd1cmUgY2FwdGlvbjpcbntzaG9ydGVuX2NhcHRpb24ob2JqWyd2YWx1ZSddKX0ifV0KCgpkZWYgYnVpbGRfbWVzc2FnZXMocm93LCBzd2FwPUZhbHNlKToKICAgIG9ial8xLCBvYmpfMiA9IHJvd1sib2JqXzEiXSwgcm93WyJvYmpfMiJdCiAgICBpZiBzd2FwOgogICAgICAgIG9ial8xLCBvYmpfMiA9IG9ial8yLCBvYmpfMQogICAgY29udGVudCA9IG9iamVjdF9jb250ZW50KDEsIG9ial8xKSArIG9iamVjdF9jb250ZW50KDIsIG9ial8yKQogICAgY29udGVudC5hcHBlbmQoeyJ0eXBlIjogInRleHQiLCAidGV4dCI6ICJDbGFzc2lmeSB0aGVpciByZWxhdGlvbnNoaXAuIFJlcGx5IHdpdGggb25lIGRpZ2l0IG9ubHkuIn0pCiAgICByZXR1cm4gWwogICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IFt7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogU1lTVEVNX1BST01QVH1dfSwKICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogY29udGVudH0sCiAgICAgICAgeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogW3sidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBzdHIoaW50KHJvd1sibGFiZWwiXSkpfV19LAogICAgXQoKCmNsYXNzIE1hbmlmZXN0RGF0YXNldCh0b3JjaC51dGlscy5kYXRhLkRhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGgsIG92ZXJzYW1wbGVfc2FtZV9maWd1cmU9RmFsc2UpOgogICAgICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICBzZWxmLnJvd3MgPSBbanNvbi5sb2FkcyhsaW5lKSBmb3IgbGluZSBpbiBoYW5kbGVdCiAgICAgICAgaWYgb3ZlcnNhbXBsZV9zYW1lX2ZpZ3VyZToKICAgICAgICAgICAgY2xhc3NfemVybyA9IFtyb3cgZm9yIHJvdyBpbiBzZWxmLnJvd3MgaWYgcm93WyJsYWJlbCJdID09IDBdCiAgICAgICAgICAgIHNlbGYucm93cy5leHRlbmQoY2xhc3NfemVybykKICAgICAgICAgICAgc2VsZi5yb3dzLmV4dGVuZChjbGFzc196ZXJvKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5yb3dzKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleCk6CiAgICAgICAgcm93ID0gZGljdChzZWxmLnJvd3NbaW5kZXhdKQogICAgICAgIHJvd1siX3N3YXAiXSA9IHJhbmRvbS5yYW5kb20oKSA8IDAuNQogICAgICAgIHJldHVybiByb3cKCgpjbGFzcyBMYWJlbE9ubHlDb2xsYXRvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwcm9jZXNzb3IpOgogICAgICAgIHNlbGYucHJvY2Vzc29yID0gcHJvY2Vzc29yCiAgICAgICAgc2VsZi5sYWJlbF90b2tlbl9pZHMgPSBbXQogICAgICAgIGZvciBkaWdpdCBpbiAiMDEyMyI6CiAgICAgICAgICAgIGlkcyA9IHByb2Nlc3Nvci50b2tlbml6ZXIuZW5jb2RlKGRpZ2l0LCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICAgICAgICAgIGlmIGxlbihpZHMpICE9IDE6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTGFiZWwge2RpZ2l0fSBpcyBub3QgYSBzaW5nbGUgdG9rZW46IHtpZHN9IikKICAgICAgICAgICAgc2VsZi5sYWJlbF90b2tlbl9pZHMuYXBwZW5kKGlkc1swXSkKCiAgICBkZWYgX19jYWxsX18oc2VsZiwgZmVhdHVyZXMpOgogICAgICAgIGlmIGxlbihmZWF0dXJlcykgIT0gMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHBlci1kZXZpY2UgYmF0Y2ggMSwgcmVjZWl2ZWQge2xlbihmZWF0dXJlcyl9IikKICAgICAgICByb3cgPSBmZWF0dXJlc1swXQogICAgICAgIG1lc3NhZ2VzID0gYnVpbGRfbWVzc2FnZXMocm93LCBzd2FwPXJvdy5nZXQoIl9zd2FwIiwgRmFsc2UpKQogICAgICAgIGJhdGNoID0gc2VsZi5wcm9jZXNzb3IuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXMsCiAgICAgICAgICAgIHRva2VuaXplPVRydWUsCiAgICAgICAgICAgIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1GYWxzZSwKICAgICAgICAgICAgcmV0dXJuX2RpY3Q9VHJ1ZSwKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICAgICApCiAgICAgICAgdGFyZ2V0X2lkID0gc2VsZi5sYWJlbF90b2tlbl9pZHNbaW50KHJvd1sibGFiZWwiXSldCiAgICAgICAgcG9zaXRpb25zID0gdG9yY2gud2hlcmUoYmF0Y2hbImlucHV0X2lkcyJdWzBdID09IHRhcmdldF9pZClbMF0KICAgICAgICBpZiBub3QgbGVuKHBvc2l0aW9ucyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQXNzaXN0YW50IGxhYmVsIHRva2VuIG5vdCBmb3VuZC4iKQogICAgICAgIGxhYmVscyA9IHRvcmNoLmZ1bGxfbGlrZShiYXRjaFsiaW5wdXRfaWRzIl0sIC0xMDApCiAgICAgICAgbGFiZWxzWzAsIGludChwb3NpdGlvbnNbLTFdKV0gPSB0YXJnZXRfaWQKICAgICAgICBiYXRjaFsibGFiZWxzIl0gPSBsYWJlbHMKICAgICAgICByZXR1cm4gYmF0Y2gKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10cmFpbi1tYW5pZmVzdCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFkYXB0ZXItZGlyIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td29yay1yb290IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtcGF0aCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS4wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncmFkaWVudC1hY2N1bXVsYXRpb24iLCB0eXBlPWludCwgZGVmYXVsdD04KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdmVyc2FtcGxlLXNhbWUtZmlndXJlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2F2ZS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgICMgU2VsZWN0IHRoaXMgcmFuaydzIEdQVSBiZWZvcmUgVHJhbnNmb3JtZXJzL3RvcmNoYW8gY2FuIHByb2JlIGFuZCBpbml0aWFsaXplIENVREEuCiAgICBsb2NhbF9yYW5rID0gaW50KG9zLmVudmlyb24uZ2V0KCJMT0NBTF9SQU5LIiwgIjAiKSkKICAgIHRvcmNoLmN1ZGEuc2V0X2RldmljZShsb2NhbF9yYW5rKQoKICAgICMgVGhlc2UgaW1wb3J0cyBoYXBwZW4gb25seSBhZnRlciBhY2NlbGVyYXRlIGhhcyBjcmVhdGVkIGNsZWFuIHdvcmtlciBwcm9jZXNzZXMuCiAgICBmcm9tIHBlZnQgaW1wb3J0IExvcmFDb25maWcsIGdldF9wZWZ0X21vZGVsLCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgKAogICAgICAgIEF1dG9Qcm9jZXNzb3IsCiAgICAgICAgQml0c0FuZEJ5dGVzQ29uZmlnLAogICAgICAgIFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24sCiAgICAgICAgVHJhaW5lciwKICAgICAgICBUcmFpbmluZ0FyZ3VtZW50cywKICAgICkKCiAgICByYW5kb20uc2VlZChTRUVEICsgbG9jYWxfcmFuaykKICAgIG5wLnJhbmRvbS5zZWVkKFNFRUQgKyBsb2NhbF9yYW5rKQogICAgdG9yY2gubWFudWFsX3NlZWQoU0VFRCArIGxvY2FsX3JhbmspCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQoKICAgIHByb2Nlc3NvciA9IEF1dG9Qcm9jZXNzb3IuZnJvbV9wcmV0cmFpbmVkKGFyZ3MubW9kZWxfcGF0aCwgbWluX3BpeGVscz1NSU5fUElYRUxTLCBtYXhfcGl4ZWxzPU1BWF9QSVhFTFMpCiAgICBwcm9jZXNzb3IudG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJyaWdodCIKICAgIHF1YW50aXphdGlvbiA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsCiAgICApCiAgICBtb2RlbCA9IFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIGFyZ3MubW9kZWxfcGF0aCwKICAgICAgICBxdWFudGl6YXRpb25fY29uZmlnPXF1YW50aXphdGlvbiwKICAgICAgICBkdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIGF0dG5faW1wbGVtZW50YXRpb249InNkcGEiLAogICAgICAgIGRldmljZV9tYXA9eyIiOiBsb2NhbF9yYW5rfSwKICAgICkKICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQogICAgbW9kZWwgPSBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nKG1vZGVsLCB1c2VfZ3JhZGllbnRfY2hlY2twb2ludGluZz1UcnVlKQogICAgbGFuZ3VhZ2VfYXR0ZW50aW9uX3RhcmdldHMgPSBbInFfcHJvaiIsICJrX3Byb2oiLCAidl9wcm9qIiwgIm9fcHJvaiJdCiAgICBtb2RlbCA9IGdldF9wZWZ0X21vZGVsKAogICAgICAgIG1vZGVsLAogICAgICAgIExvcmFDb25maWcoCiAgICAgICAgICAgIHI9MTYsCiAgICAgICAgICAgIGxvcmFfYWxwaGE9MzIsCiAgICAgICAgICAgIGxvcmFfZHJvcG91dD0wLjA1LAogICAgICAgICAgICBiaWFzPSJub25lIiwKICAgICAgICAgICAgdGFza190eXBlPSJDQVVTQUxfTE0iLAogICAgICAgICAgICB0YXJnZXRfbW9kdWxlcz1sYW5ndWFnZV9hdHRlbnRpb25fdGFyZ2V0cywKICAgICAgICApLAogICAgKQogICAgdHJhaW5hYmxlX25hbWVzID0gW25hbWUgZm9yIG5hbWUsIHBhcmFtZXRlciBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWRdCiAgICBpZiBub3QgdHJhaW5hYmxlX25hbWVzOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm8gdHJhaW5hYmxlIExvUkEgcGFyYW1ldGVycyB3ZXJlIGNyZWF0ZWQuIikKICAgIHVuZXhwZWN0ZWQgPSBbbmFtZSBmb3IgbmFtZSBpbiB0cmFpbmFibGVfbmFtZXMgaWYgIi52aXN1YWwuIiBpbiBuYW1lXQogICAgaWYgdW5leHBlY3RlZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJMYW5ndWFnZS1vbmx5IHJ1biBzZWxlY3RlZCB2aXN1YWwgcGFyYW1ldGVyczoge3VuZXhwZWN0ZWR9IikKICAgIG1pc3NpbmcgPSBbCiAgICAgICAgc3VmZml4IGZvciBzdWZmaXggaW4gbGFuZ3VhZ2VfYXR0ZW50aW9uX3RhcmdldHMKICAgICAgICBpZiBub3QgYW55KGYiLntzdWZmaXh9LiIgaW4gbmFtZSBmb3IgbmFtZSBpbiB0cmFpbmFibGVfbmFtZXMpCiAgICBdCiAgICBpZiBtaXNzaW5nOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIk1pc3NpbmcgbGFuZ3VhZ2UtYXR0ZW50aW9uIExvUkEgdGFyZ2V0czoge21pc3Npbmd9IikKICAgIGlmIGxvY2FsX3JhbmsgPT0gMDoKICAgICAgICBwcmludCgiTG9SQSB2YXJpYW50OiBRd2VuMy1WTC04QiBsYW5ndWFnZSBhdHRlbnRpb24gb25seSIsIGZsdXNoPVRydWUpCiAgICAgICAgbW9kZWwucHJpbnRfdHJhaW5hYmxlX3BhcmFtZXRlcnMoKQogICAgICAgIHByaW50KGYiVHJhaW5hYmxlIHBhcmFtZXRlciB0ZW5zb3JzOiB7bGVuKHRyYWluYWJsZV9uYW1lcyl9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBmb3IgbmFtZSBpbiB0cmFpbmFibGVfbmFtZXM6CiAgICAgICAgICAgIHByaW50KGYiICBUUkFJTkFCTEUge25hbWV9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBwcmludChmIk1vZGVsIGxvYWQgb24gcmFuayAwOiB7KHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAvIDYwOi4yZn0gbWluIiwgZmx1c2g9VHJ1ZSkKCiAgICBkYXRhc2V0ID0gTWFuaWZlc3REYXRhc2V0KGFyZ3MudHJhaW5fbWFuaWZlc3QsIG92ZXJzYW1wbGVfc2FtZV9maWd1cmU9YXJncy5vdmVyc2FtcGxlX3NhbWVfZmlndXJlKQogICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgIHByaW50KGYiRWZmZWN0aXZlIHRyYWluaW5nIHJvd3M6IHtsZW4oZGF0YXNldCl9IiwgZmx1c2g9VHJ1ZSkKICAgIHRyYWluaW5nX2FyZ3MgPSBUcmFpbmluZ0FyZ3VtZW50cygKICAgICAgICBvdXRwdXRfZGlyPXN0cihQYXRoKGFyZ3Mud29ya19yb290KSAvICJ0cmFpbmVyX291dHB1dCIpLAogICAgICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZT0xLAogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz1hcmdzLmdyYWRpZW50X2FjY3VtdWxhdGlvbiwKICAgICAgICBudW1fdHJhaW5fZXBvY2hzPWFyZ3MuZXBvY2hzLAogICAgICAgIGxlYXJuaW5nX3JhdGU9MWUtNCwKICAgICAgICB3YXJtdXBfcmF0aW89MC4wNSwKICAgICAgICBscl9zY2hlZHVsZXJfdHlwZT0iY29zaW5lIiwKICAgICAgICB3ZWlnaHRfZGVjYXk9MC4wMSwKICAgICAgICBtYXhfZ3JhZF9ub3JtPTEuMCwKICAgICAgICBmcDE2PVRydWUsCiAgICAgICAgYmYxNj1GYWxzZSwKICAgICAgICBncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUsCiAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9LAogICAgICAgIG9wdGltPSJwYWdlZF9hZGFtd184Yml0IiwKICAgICAgICBsb2dnaW5nX3N0ZXBzPTEwLAogICAgICAgIGV2YWxfc3RyYXRlZ3k9Im5vIiwKICAgICAgICBzYXZlX3N0cmF0ZWd5PSJzdGVwcyIgaWYgYXJncy5zYXZlX3N0ZXBzID4gMCBlbHNlICJubyIsCiAgICAgICAgc2F2ZV9zdGVwcz1tYXgoMSwgYXJncy5zYXZlX3N0ZXBzKSwKICAgICAgICBzYXZlX3RvdGFsX2xpbWl0PTIsCiAgICAgICAgcmVwb3J0X3RvPSJub25lIiwKICAgICAgICByZW1vdmVfdW51c2VkX2NvbHVtbnM9RmFsc2UsCiAgICAgICAgZGF0YWxvYWRlcl9udW1fd29ya2Vycz0wLAogICAgICAgIGRkcF9maW5kX3VudXNlZF9wYXJhbWV0ZXJzPUZhbHNlLAogICAgICAgIHNlZWQ9U0VFRCwKICAgICkKICAgIHRyYWluZXIgPSBUcmFpbmVyKAogICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgIGFyZ3M9dHJhaW5pbmdfYXJncywKICAgICAgICB0cmFpbl9kYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgZGF0YV9jb2xsYXRvcj1MYWJlbE9ubHlDb2xsYXRvcihwcm9jZXNzb3IpLAogICAgKQogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICB0cmFpbl9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgcmVzdWx0ID0gdHJhaW5lci50cmFpbigpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdHJhaW5fc3RhcnRlZAoKICAgIGlmIHRyYWluZXIuaXNfd29ybGRfcHJvY2Vzc196ZXJvKCk6CiAgICAgICAgYWRhcHRlcl9wYXRoID0gUGF0aChhcmdzLmFkYXB0ZXJfZGlyKQogICAgICAgIGFkYXB0ZXJfcGF0aC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKGFkYXB0ZXJfcGF0aCkKICAgICAgICBwcm9jZXNzb3Iuc2F2ZV9wcmV0cmFpbmVkKGFkYXB0ZXJfcGF0aCkKICAgICAgICBtZXRyaWNzID0gZGljdChyZXN1bHQubWV0cmljcykKICAgICAgICBtZXRyaWNzLnVwZGF0ZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgIndhbGxfc2Vjb25kcyI6IGVsYXBzZWQsCiAgICAgICAgICAgICAgICAid2FsbF9taW51dGVzIjogZWxhcHNlZCAvIDYwLAogICAgICAgICAgICAgICAgIm9wdGltaXplcl9zdGVwcyI6IGludCh0cmFpbmVyLnN0YXRlLmdsb2JhbF9zdGVwKSwKICAgICAgICAgICAgICAgICJzZWNvbmRzX3Blcl9vcHRpbWl6ZXJfc3RlcCI6IGVsYXBzZWQgLyBtYXgoMSwgdHJhaW5lci5zdGF0ZS5nbG9iYWxfc3RlcCksCiAgICAgICAgICAgICAgICAicGVha19ncHVfZ2liX3JhbmswIjogdG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpIC8gMioqMzAsCiAgICAgICAgICAgICAgICAiZWZmZWN0aXZlX3RyYWluaW5nX3Jvd3MiOiBsZW4oZGF0YXNldCksCiAgICAgICAgICAgICAgICAibW9kZWxfdmFyaWFudCI6ICJRd2VuMy1WTC04Qi1JbnN0cnVjdCIsCiAgICAgICAgICAgICAgICAibG9yYV92YXJpYW50IjogImxhbmd1YWdlX2F0dGVudGlvbl9vbmx5IiwKICAgICAgICAgICAgICAgICJsb3JhX3RhcmdldHMiOiBsYW5ndWFnZV9hdHRlbnRpb25fdGFyZ2V0cywKICAgICAgICAgICAgICAgICJ0cmFpbmFibGVfcGFyYW1ldGVyX3RlbnNvcnMiOiB0cmFpbmFibGVfbmFtZXMsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICAgICAgd2l0aCAoYWRhcHRlcl9wYXRoIC8gInRyYWluaW5nX21ldHJpY3MuanNvbiIpLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIGpzb24uZHVtcChtZXRyaWNzLCBoYW5kbGUsIGluZGVudD0yKQogICAgICAgIHByaW50KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'))
train_command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--multi_gpu', '--num_processes', '2',
    str(TRAIN_SCRIPT_PATH),
    '--train-manifest', str(TRAIN_MANIFEST),
    '--model-path', MODEL_PATH,
    '--adapter-dir', str(ADAPTER_DIR),
    '--work-root', str(WORK_ROOT),
    '--epochs', str(NUM_EPOCHS),
    '--gradient-accumulation', str(GRADIENT_ACCUMULATION),
    '--save-steps', '100',
]
if OVERSAMPLE_SAME_FIGURE:
    train_command.append('--oversample-same-figure')
print('Launching:', ' '.join(train_command), flush=True)
training_launch_started = time.perf_counter()
launch_env = dict(
    os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
)
subprocess.run(train_command, check=True, env=launch_env)
print(f'Total DDP training launch: {(time.perf_counter() - training_launch_started) / 60:.2f} min')

## Optional two-GPU test inference

Each GPU processes 5,000 rows and writes probabilities independently. Inference reads only the four valid next-token logits; it does not generate a long response. Keep swap TTA disabled unless a validation experiment justifies doubling inference time.


In [ ]:
def inference_worker(manifest_path, model_path, adapter_path, output_dir, test_limit, swap_tta, modality_mask):
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
    from peft import PeftModel

    rank = int(os.environ.get('LOCAL_RANK', '0'))
    world_size = int(os.environ.get('WORLD_SIZE', '2'))
    torch.cuda.set_device(rank)
    with Path(manifest_path).open('r', encoding='utf-8') as handle:
        rows = [json.loads(line) for line in handle]
    if test_limit is not None:
        rows = rows[:test_limit]
    rows = rows[rank::world_size]

    processor = AutoProcessor.from_pretrained(adapter_path, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        model_path, quantization_config=quantization, dtype=torch.float16,
        attn_implementation='sdpa', device_map={'': rank},
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval(); model.config.use_cache = True
    token_ids = []
    for digit in '0123':
        ids = processor.tokenizer.encode(digit, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f'Label {digit} is not one token: {ids}')
        token_ids.append(ids[0])

    @torch.inference_mode()
    def predict(row, swap=False):
        batch = processor.apply_chat_template(
            build_messages(row, include_answer=False, swap=swap),
            tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt',
        )
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        logits = model(**batch).logits[0, -1, token_ids].float()
        if modality_mask and row['modality'] in {'CC', 'II'}:
            logits[0] = float('-inf')
        return torch.softmax(logits, dim=-1).cpu().numpy()

    output_path = Path(output_dir) / f'probabilities_rank{rank}.csv'
    started = time.perf_counter()
    with output_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['id', *[f'p_{x}' for x in TARGET_COLUMNS]])
        writer.writeheader()
        for index, row in enumerate(rows, start=1):
            p = predict(row)
            if swap_tta:
                p = 0.5 * (p + predict(row, swap=True))
            writer.writerow({'id': row['id'], **{f'p_{name}': float(p[i]) for i, name in enumerate(TARGET_COLUMNS)}})
            if index % 100 == 0:
                elapsed = time.perf_counter() - started
                print(f'rank={rank} {index}/{len(rows)} {elapsed / index:.3f}s/row ETA={(elapsed / index) * (len(rows)-index) / 3600:.2f}h')
    elapsed = time.perf_counter() - started
    print(f'Rank {rank} finished {len(rows)} rows in {elapsed / 3600:.2f}h')

if RUN_TEST_INFERENCE:
    INFERENCE_SCRIPT_PATH = WORK_ROOT / 'infer_ddp.py'
    INFERENCE_SCRIPT_PATH.write_bytes(base64.b64decode('aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UsIEltYWdlRmlsZQoKSW1hZ2VGaWxlLkxPQURfVFJVTkNBVEVEX0lNQUdFUyA9IFRydWUKCk1JTl9QSVhFTFMgPSAyNTYgKiAyNTYKTUFYX1BJWEVMUyA9IDQ0OCAqIDQ0OApNQVhfVEVYVF9DSEFSUyA9IDMwMDAKVEFSR0VUX0NPTFVNTlMgPSBbInNhbWVfZmlndXJlIiwgInNhbWVfcGFwZXIiLCAicmVsYXRlZF9wYXBlcnMiLCAidW5yZWxhdGVkX3BhcGVycyJdClNZU1RFTV9QUk9NUFQgPSAiIiJZb3UgY2xhc3NpZnkgdGhlIHJlbGF0aW9uc2hpcCBiZXR3ZWVuIHR3byBvYmplY3RzIGZyb20gYXN0cm9ub215IHBhcGVycy4KMDogVGhlIG9iamVjdHMgYXJlIHRoZSBmaWd1cmUgYW5kIGNhcHRpb24gb2YgdGhlIHNhbWUgc2NpZW50aWZpYyBmaWd1cmUuCjE6IFRoZSBvYmplY3RzIGFyZSBmcm9tIGRpZmZlcmVudCBmaWd1cmVzIGluIHRoZSBzYW1lIHBhcGVyLgoyOiBUaGUgb2JqZWN0cyBhcmUgZnJvbSBkaWZmZXJlbnQgcGFwZXJzIGFuZCBvbmUgcGFwZXIgY2l0ZXMgdGhlIG90aGVyLgozOiBUaGUgb2JqZWN0cyBhcmUgZnJvbSB1bnJlbGF0ZWQgcGFwZXJzLgpUaGUgcmVsYXRpb25zaGlwIGlzIHN5bW1ldHJpYy4gT3V0cHV0IG9ubHkgb25lIGRpZ2l0OiAwLCAxLCAyLCBvciAzLiIiIi5zdHJpcCgpCgoKZGVmIHNob3J0ZW5fY2FwdGlvbih0ZXh0LCBtYXhfY2hhcnM9TUFYX1RFWFRfQ0hBUlMpOgogICAgaWYgbGVuKHRleHQpIDw9IG1heF9jaGFyczoKICAgICAgICByZXR1cm4gdGV4dAogICAgaGFsZiA9IG1heF9jaGFycyAvLyAyCiAgICByZXR1cm4gdGV4dFs6aGFsZl0gKyAiXG5bLi4ubWlkZGxlIHRydW5jYXRlZC4uLl1cbiIgKyB0ZXh0Wy1oYWxmOl0KCgpkZWYgb2JqZWN0X2NvbnRlbnQobnVtYmVyLCBvYmopOgogICAgaWYgb2JqWyJraW5kIl0gPT0gImltYWdlIjoKICAgICAgICB3aXRoIEltYWdlLm9wZW4ob2JqWyJ2YWx1ZSJdKSBhcyBzb3VyY2U6CiAgICAgICAgICAgIGltYWdlID0gc291cmNlLmNvbnZlcnQoIlJHQiIpCiAgICAgICAgcmV0dXJuIFsKICAgICAgICAgICAgeyJ0eXBlIjogInRleHQiLCAidGV4dCI6IGYiT2JqZWN0IHtudW1iZXJ9IGlzIGEgc2NpZW50aWZpYyBmaWd1cmU6In0sCiAgICAgICAgICAgIHsidHlwZSI6ICJpbWFnZSIsICJpbWFnZSI6IGltYWdlfSwKICAgICAgICBdCiAgICByZXR1cm4gW3sidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBmIk9iamVjdCB7bnVtYmVyfSBpcyBhIGZpZ3VyZSBjYXB0aW9uOlxue3Nob3J0ZW5fY2FwdGlvbihvYmpbJ3ZhbHVlJ10pfSJ9XQoKCmRlZiBidWlsZF9tZXNzYWdlcyhyb3csIHN3YXA9RmFsc2UpOgogICAgb2JqXzEsIG9ial8yID0gcm93WyJvYmpfMSJdLCByb3dbIm9ial8yIl0KICAgIGlmIHN3YXA6CiAgICAgICAgb2JqXzEsIG9ial8yID0gb2JqXzIsIG9ial8xCiAgICBjb250ZW50ID0gb2JqZWN0X2NvbnRlbnQoMSwgb2JqXzEpICsgb2JqZWN0X2NvbnRlbnQoMiwgb2JqXzIpCiAgICBjb250ZW50LmFwcGVuZCh7InR5cGUiOiAidGV4dCIsICJ0ZXh0IjogIkNsYXNzaWZ5IHRoZWlyIHJlbGF0aW9uc2hpcC4gUmVwbHkgd2l0aCBvbmUgZGlnaXQgb25seS4ifSkKICAgIHJldHVybiBbCiAgICAgICAgeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50IjogW3sidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBTWVNURU1fUFJPTVBUfV19LAogICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBjb250ZW50fSwKICAgIF0KCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10ZXN0LW1hbmlmZXN0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtcGF0aCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFkYXB0ZXItZGlyIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRlc3QtbGltaXQiLCB0eXBlPWludCwgZGVmYXVsdD0tMSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3dhcC10dGEiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb2RhbGl0eS1tYXNrIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgIyBTZWxlY3QgdGhpcyByYW5rJ3MgR1BVIGJlZm9yZSBUcmFuc2Zvcm1lcnMvdG9yY2hhbyBjYW4gcHJvYmUgYW5kIGluaXRpYWxpemUgQ1VEQS4KICAgIHJhbmsgPSBpbnQob3MuZW52aXJvbi5nZXQoIkxPQ0FMX1JBTksiLCAiMCIpKQogICAgd29ybGRfc2l6ZSA9IGludChvcy5lbnZpcm9uLmdldCgiV09STERfU0laRSIsICIyIikpCiAgICB0b3JjaC5jdWRhLnNldF9kZXZpY2UocmFuaykKCiAgICAjIEltcG9ydHMgb2NjdXIgaW4gZnJlc2ggYWNjZWxlcmF0ZSB3b3JrZXJzLCBuZXZlciBpbiBhIGZvcmsgb2YgYSBDVURBLWluaXRpYWxpemVkIGtlcm5lbC4KICAgIGZyb20gcGVmdCBpbXBvcnQgUGVmdE1vZGVsCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Byb2Nlc3NvciwgQml0c0FuZEJ5dGVzQ29uZmlnLCBRd2VuM1ZMRm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uCgogICAgd2l0aCBQYXRoKGFyZ3MudGVzdF9tYW5pZmVzdCkub3BlbigiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICByb3dzID0gW2pzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gaGFuZGxlXQogICAgaWYgYXJncy50ZXN0X2xpbWl0ID49IDA6CiAgICAgICAgcm93cyA9IHJvd3NbOiBhcmdzLnRlc3RfbGltaXRdCiAgICByb3dzID0gcm93c1tyYW5rOjp3b3JsZF9zaXplXQoKICAgIHByb2Nlc3NvciA9IEF1dG9Qcm9jZXNzb3IuZnJvbV9wcmV0cmFpbmVkKGFyZ3MuYWRhcHRlcl9kaXIsIG1pbl9waXhlbHM9TUlOX1BJWEVMUywgbWF4X3BpeGVscz1NQVhfUElYRUxTKQogICAgcXVhbnRpemF0aW9uID0gQml0c0FuZEJ5dGVzQ29uZmlnKAogICAgICAgIGxvYWRfaW5fNGJpdD1UcnVlLAogICAgICAgIGJuYl80Yml0X3F1YW50X3R5cGU9Im5mNCIsCiAgICAgICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudD1UcnVlLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICkKICAgIGJhc2UgPSBRd2VuM1ZMRm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBhcmdzLm1vZGVsX3BhdGgsCiAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1xdWFudGl6YXRpb24sCiAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICBhdHRuX2ltcGxlbWVudGF0aW9uPSJzZHBhIiwKICAgICAgICBkZXZpY2VfbWFwPXsiIjogcmFua30sCiAgICApCiAgICBtb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZSwgYXJncy5hZGFwdGVyX2RpcikKICAgIG1vZGVsLmV2YWwoKQogICAgbW9kZWwuY29uZmlnLnVzZV9jYWNoZSA9IFRydWUKICAgIHRva2VuX2lkcyA9IFtdCiAgICBmb3IgZGlnaXQgaW4gIjAxMjMiOgogICAgICAgIGlkcyA9IHByb2Nlc3Nvci50b2tlbml6ZXIuZW5jb2RlKGRpZ2l0LCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICAgICAgaWYgbGVuKGlkcykgIT0gMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkxhYmVsIHtkaWdpdH0gaXMgbm90IG9uZSB0b2tlbjoge2lkc30iKQogICAgICAgIHRva2VuX2lkcy5hcHBlbmQoaWRzWzBdKQoKICAgIEB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpCiAgICBkZWYgcHJlZGljdChyb3csIHN3YXA9RmFsc2UpOgogICAgICAgIGJhdGNoID0gcHJvY2Vzc29yLmFwcGx5X2NoYXRfdGVtcGxhdGUoCiAgICAgICAgICAgIGJ1aWxkX21lc3NhZ2VzKHJvdywgc3dhcD1zd2FwKSwKICAgICAgICAgICAgdG9rZW5pemU9VHJ1ZSwKICAgICAgICAgICAgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsCiAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsCiAgICAgICAgICAgIHJldHVybl90ZW5zb3JzPSJwdCIsCiAgICAgICAgKQogICAgICAgIGJhdGNoID0ge2tleTogdmFsdWUudG8obW9kZWwuZGV2aWNlKSBpZiB0b3JjaC5pc190ZW5zb3IodmFsdWUpIGVsc2UgdmFsdWUgZm9yIGtleSwgdmFsdWUgaW4gYmF0Y2guaXRlbXMoKX0KICAgICAgICBsb2dpdHMgPSBtb2RlbCgqKmJhdGNoKS5sb2dpdHNbMCwgLTEsIHRva2VuX2lkc10uZmxvYXQoKQogICAgICAgIGlmIGFyZ3MubW9kYWxpdHlfbWFzayBhbmQgcm93WyJtb2RhbGl0eSJdIGluIHsiQ0MiLCAiSUkifToKICAgICAgICAgICAgbG9naXRzWzBdID0gZmxvYXQoIi1pbmYiKQogICAgICAgIHJldHVybiB0b3JjaC5zb2Z0bWF4KGxvZ2l0cywgZGltPS0xKS5jcHUoKS5udW1weSgpCgogICAgb3V0cHV0X2RpciA9IFBhdGgoYXJncy5vdXRwdXRfZGlyKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXRwdXRfcGF0aCA9IG91dHB1dF9kaXIgLyBmInByb2JhYmlsaXRpZXNfcmFua3tyYW5rfS5jc3YiCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgd2l0aCBvdXRwdXRfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgaGFuZGxlOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1bImlkIiwgKltmInBfe25hbWV9IiBmb3IgbmFtZSBpbiBUQVJHRVRfQ09MVU1OU11dKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIGluZGV4LCByb3cgaW4gZW51bWVyYXRlKHJvd3MsIHN0YXJ0PTEpOgogICAgICAgICAgICBwcm9iYWJpbGl0aWVzID0gcHJlZGljdChyb3cpCiAgICAgICAgICAgIGlmIGFyZ3Muc3dhcF90dGE6CiAgICAgICAgICAgICAgICBwcm9iYWJpbGl0aWVzID0gMC41ICogKHByb2JhYmlsaXRpZXMgKyBwcmVkaWN0KHJvdywgc3dhcD1UcnVlKSkKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KAogICAgICAgICAgICAgICAgeyJpZCI6IHJvd1siaWQiXSwgKip7ZiJwX3tuYW1lfSI6IGZsb2F0KHByb2JhYmlsaXRpZXNbaV0pIGZvciBpLCBuYW1lIGluIGVudW1lcmF0ZShUQVJHRVRfQ09MVU1OUyl9fQogICAgICAgICAgICApCiAgICAgICAgICAgIGlmIGluZGV4ICUgMTAwID09IDA6CiAgICAgICAgICAgICAgICBlbGFwc2VkID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKICAgICAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgICAgIGYicmFuaz17cmFua30ge2luZGV4fS97bGVuKHJvd3MpfSB7ZWxhcHNlZC9pbmRleDouM2Z9cy9yb3cgIgogICAgICAgICAgICAgICAgICAgIGYiRVRBPXsoZWxhcHNlZC9pbmRleCkqKGxlbihyb3dzKS1pbmRleCkvMzYwMDouMmZ9aCIsCiAgICAgICAgICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAogICAgcHJpbnQoZiJSYW5rIHtyYW5rfSBmaW5pc2hlZCB7bGVuKHJvd3MpfSByb3dzIGluIHtlbGFwc2VkLzM2MDA6LjJmfWgiLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'))
    inference_command = [
        sys.executable, '-m', 'accelerate.commands.launch',
        '--multi_gpu', '--num_processes', '2',
        str(INFERENCE_SCRIPT_PATH),
        '--test-manifest', str(TEST_MANIFEST),
        '--model-path', MODEL_PATH,
        '--adapter-dir', str(ADAPTER_DIR),
        '--output-dir', str(PREDICTION_DIR),
        '--test-limit', str(-1 if TEST_LIMIT is None else TEST_LIMIT),
    ]
    if USE_SWAP_TTA:
        inference_command.append('--swap-tta')
    if APPLY_MODALITY_MASK:
        inference_command.append('--modality-mask')
    print('Launching:', ' '.join(inference_command), flush=True)
    inference_started = time.perf_counter()
    subprocess.run(inference_command, check=True, env=launch_env)
    print(f'Two-GPU inference launch: {(time.perf_counter() - inference_started) / 3600:.2f}h')

## Merge shards and create submission

In [ ]:
if RUN_TEST_INFERENCE:
    shard_paths = [PREDICTION_DIR / f'probabilities_rank{rank}.csv' for rank in range(2)]
    probabilities = pd.concat([pd.read_csv(path, dtype={'id': str}) for path in shard_paths], ignore_index=True)
    with TEST_MANIFEST.open('r', encoding='utf-8') as handle:
        ordered_ids = [json.loads(line)['id'] for line in handle]
    if TEST_LIMIT is not None:
        ordered_ids = ordered_ids[:TEST_LIMIT]
    order = {str(row_id): index for index, row_id in enumerate(ordered_ids)}
    probabilities['order'] = probabilities['id'].map(order)
    assert probabilities['order'].notna().all()
    probabilities = probabilities.sort_values('order').drop(columns='order').reset_index(drop=True)
    assert probabilities['id'].is_unique and len(probabilities) == len(ordered_ids)

    probability_columns = [f'p_{name}' for name in TARGET_COLUMNS]
    predictions = probabilities[probability_columns].to_numpy().argmax(axis=1)
    submission = pd.DataFrame({'id': probabilities['id']})
    for class_index, name in enumerate(TARGET_COLUMNS):
        submission[name] = (predictions == class_index).astype(int)
    assert submission.columns.tolist() == ['id', *TARGET_COLUMNS]
    assert submission['id'].is_unique
    assert submission[TARGET_COLUMNS].isin([0, 1]).all().all()
    assert (submission[TARGET_COLUMNS].sum(axis=1) == 1).all()

    probability_path = WORK_ROOT / 'submission_probabilities.csv'
    submission_path = WORK_ROOT / 'submission.csv'
    probabilities.to_csv(probability_path, index=False)
    submission.to_csv(submission_path, index=False)
    print('Rows:', len(submission))
    print('Prediction counts:', submission[TARGET_COLUMNS].sum().to_dict())
    if TEST_LIMIT is None:
        assert len(submission) == 10000
        print('Complete submission ready:', submission_path)
    else:
        print('Partial timing run only. Set TEST_LIMIT=None for a valid submission.')
    display(submission.head())